You are evaluating two candidate sentiment models against the sentiment model already running in production (the **baseline**). Your goal is to decide whether either candidate should ship.

“Ship” means: we would turn off the baseline and serve the candidate instead, based on the evidence you collect. Read `README.md` first — it defines the five concepts this lab is built on and lists your deliverables.

### Step 1 - Install the required dependencies, set up W&B and make sure the python version is 3.10 and above

In [1]:
# Run once. Skip this cell on later runs — everything is already installed.
!pip install -q wandb datasets transformers torch tqdm emoji pandas pyarrow scikit-learn pytest pyyaml


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import wandb

from datasets import load_dataset
from transformers import pipeline

SEED = 42
np.random.seed(SEED)

/home/autumnq/17-445/Lab-4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Run `wandb login` in a terminal first; this just confirms the notebook sees your key.
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/autumnq/.netrc.
wandb: Currently logged in as: autumnq (autumnq-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
!python --version

Python 3.12.9


In [5]:
# W&B config — every run logged below reuses these three values.
PROJECT = "mlip-lab4-slices-2026"
ENTITY = None  # None = your own account; set a team name only if your TA gives you one
RUN_NAME = "baseline_vs_candidate"


In [32]:
# Set True in Task G, after you freeze tests/manifest.yaml. Do not retune the gate for v2.
INCLUDE_CANDIDATE_V2 = True

# The registry lives in lab_helpers.py so this notebook and the pytest gate can never
# disagree about which model is which. `csv_col` is that model's column prefix in tweets.csv.
from lab_helpers import BASELINE, MODELS as MODEL_REGISTRY

MODELS = {
    name: spec
    for name, spec in MODEL_REGISTRY.items()
    if INCLUDE_CANDIDATE_V2 or name != "candidate_v2"
}
pd.DataFrame(MODELS).T

,hf_id,csv_col
baseline,cardiffnlp/twitter-roberta-base-sentiment-latest,roberta
candidate_v1,LYTinn/finetuning-sentiment-model-tweet-gpt2,gpt2
candidate_v2,Elron/deberta-v3-large-sentiment,deberta


In [7]:
# Label normalization for tweet_eval (0/1/2 -> string labels)
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

# Many HF sentiment models output labels like LABEL_0 / LABEL_1 / LABEL_2
HF_LABEL_MAP = {
    "LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive",
    "NEG": "negative", "NEU": "neutral", "POS": "positive",
    "0": "negative", "1": "neutral", "2": "positive",
}

USE_HF_DATASET = False  # set True to load tweet_eval from Hugging Face and run inference

### Step 2 - Load a dataset from Hugging Face

In [8]:
if USE_HF_DATASET:
    ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")
    df = pd.DataFrame(ds["test"]).head(500).copy()
    df["label"] = df["label"].map(ID2LABEL)
    df = df[["text", "label"]].dropna().reset_index(drop=True)
else:
    df = pd.read_csv("tweets.csv")
    # Keep prediction columns (roberta, gpt2, …) so Step 4 can skip live inference.
    df = df.rename(columns={c: c.strip() for c in df.columns})
    assert {"text", "label"}.issubset(df.columns), "tweets.csv must include text,label"
    df["label"] = df["label"].astype(str).str.lower()
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)

df.head(3)


,text,label,roberta,roberta_score,input_length,gpt2,gpt2_score,roberta_latency_ms,gpt2_latency_ms,deberta,deberta_score,deberta_latency_ms
0,@user @user what do these '1/2 naked pics' hav...,neutral,negative,0.804726,96,positive,0.913451,3004.968000,3267.502791,negative,0.667068,289.676375
1,OH: “I had a blue penis while I was this” [pla...,neutral,neutral,0.866949,72,neutral,0.753405,44.326958,39.965208,neutral,0.723149,240.529875
2,"@user @user That's coming, but I think the vic...",neutral,neutral,0.763724,87,positive,0.999962,36.559709,33.581416,negative,0.687591,229.930750


### Step 3 - Define Failure-Relevant Metadata

#TODO:
In this step, you will create **at least 5** metadata columns that help you slice and analyze model behavior in Weights & Biases (W&B).
These metadata columns should **capture meaningful properties of the data or model behavior that may influence performance**. You can define them using:

1. Value matching (e.g., tweets containing hashtags or mentions)
2. Regex patterns (e.g., negation words, strong sentiment terms like love or hate)
3. Heuristics (e.g., emoji count, all-caps text, tweet length buckets)

Each metadata column should correspond to a potential hypothesis about when or why a model might succeed or fail.
These columns will be propagated through inference and included in the final predictions_table logged to W&B.

After inference, your W&B table (df_long) will contain:
- The original tweet text
- Ground-truth sentiment labels
- Model predictions and confidence scores
- All metadata columns you defined for slicing

You will use these metadata fields in the W&B UI (via the ➕ Filter option) to:
- Create slices of the data
- Compare model behavior across slices
- Identify patterns, weaknesses, or regressions that are not visible in overall accuracy

In [9]:
# Step 3 – Add slicing metadata (text-only)
#
# TODO: add your own hypothesis-driven metadata here.
# Edit slices.py (META_COLS, add_metadata, and get_slices). Here are examples
# of the kinds of metadata columns you can add & analyse.
# Categories you can explore are: Linguistic, Emotional/semantic, Model-behavioral.
# Do not reuse the ones given below.

from slices import META_COLS, add_metadata, get_slices

assert len(META_COLS) >= 5
df = add_metadata(df)
df[META_COLS].head(3)

,emoji_count,has_hashtag,has_mention,has_negation,length_bucket,is_all_caps,has_question_mark,has_multiple_sentences,has_exclamation_mark,has_quote
0,0,False,True,True,51-100,False,True,True,False,True
1,0,False,False,False,51-100,False,False,False,False,False
2,0,False,True,False,51-100,False,False,False,False,True


In [10]:
# Transformers requires a backend (PyTorch/TensorFlow/Flax). We'll use PyTorch.
try:
    import torch, transformers, sys
    print("torch:", torch.__version__)
    print("transformers:", transformers.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Python:", sys.executable)
except Exception as e:
    raise RuntimeError("Install PyTorch before proceeding: pip install torch torchvision torchaudio") from e

torch: 2.14.0+cu130
transformers: 5.17.0
CUDA available: False
Python: /home/autumnq/17-445/Lab-4/.venv/bin/python


###  Step 4 – Run Inference (Models in MODELS)

In this step, you'll score every model in `MODELS`. With `USE_HF_DATASET = False` (the default), predictions are read from `tweets.csv`. Set it to `True` to run HuggingFace inference instead.

In [33]:
from tqdm.auto import tqdm

def run_pipeline(model_id: str, texts: list[str]):
    clf = pipeline(
        "text-classification",
        model=model_id,
        truncation=True,
        max_length=128,     # avoid truncation warnings
        framework="pt",
        device=-1           # CPU
    )
    # (Optional) sanity check label mapping for this model
    # print(model_id, clf.model.config.id2label)

    preds, confs = [], []
    for t in tqdm(texts, desc=f"Infer: {model_id}"):
        out = clf(t)[0]
        lbl = HF_LABEL_MAP.get(out["label"], out["label"])
        preds.append(lbl)
        confs.append(float(out["score"]))
    return preds, confs

pred_frames = []

if not USE_HF_DATASET:
    # Build df_long from cached columns in tweets.csv (no network).
    for model_name, spec in MODELS.items():
        col = spec["csv_col"]
        score_col = f"{col}_score"
        if col not in df.columns or score_col not in df.columns:
            print(f"Skipping {model_name}: tweets.csv missing {col}/{score_col}")
            continue
        keep = ["text", "label"] + [c for c in META_COLS if c in df.columns]
        tmp = df[keep].copy()
        tmp["model"] = model_name
        tmp["pred"] = df[col].astype(str).str.lower()
        tmp["conf"] = pd.to_numeric(df[score_col], errors="coerce")
        lat_col = f"{col}_latency_ms"
        if lat_col in df.columns:
            tmp["latency_ms"] = pd.to_numeric(df[lat_col], errors="coerce")
        pred_frames.append(tmp)
else:
    texts = df["text"].tolist()
    for model_name, spec in MODELS.items():
        yhat, conf = run_pipeline(spec["hf_id"], texts)
        tmp = df.copy()
        tmp["model"] = model_name
        tmp["pred"] = yhat
        tmp["conf"] = conf
        pred_frames.append(tmp)

df_long = pd.concat(pred_frames, ignore_index=True)

# Add a stable example id so reshaping won't silently drop duplicates
df_long["ex_id"] = df_long.groupby(["text", "label"]).ngroup()

df_long.head(5)

,text,label,emoji_count,has_hashtag,has_mention,has_negation,length_bucket,is_all_caps,has_question_mark,has_multiple_sentences,has_exclamation_mark,has_quote,model,pred,conf,latency_ms,ex_id
0,@user @user what do these '1/2 naked pics' hav...,neutral,0,False,True,True,51-100,False,True,True,False,True,baseline,negative,0.804726,3004.968000,113
1,OH: “I had a blue penis while I was this” [pla...,neutral,0,False,False,False,51-100,False,False,False,False,False,baseline,neutral,0.866949,44.326958,363
2,"@user @user That's coming, but I think the vic...",neutral,0,False,True,False,51-100,False,False,False,False,True,baseline,neutral,0.763724,36.559709,102
3,I think I may be finally in with the in crowd ...,positive,0,True,True,False,51-100,False,False,False,False,False,baseline,positive,0.774047,37.761583,305
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative,0,False,True,False,101-200,False,False,True,False,False,baseline,neutral,0.416397,37.504542,160


In [34]:
# Step 4.5 – Wide-format Table for Model Comparison (Optional but recommended)
# One row per tweet, with each model's predictions in columns
# TODO: Replace with your metadata
assert len(META_COLS) >= 5
df_wide = df_long.pivot_table(
    index=["ex_id", "text", "label"] + META_COLS,
    columns="model",
    values=["pred", "conf"],
    aggfunc="first"
).reset_index()

# Flatten column names (e.g., pred_baseline, conf_candidate_v1)
df_wide.columns = ["_".join([c for c in col if c]).strip("_") for col in df_wide.columns]

df_wide.head(5)

,ex_id,text,label,emoji_count,has_hashtag,has_mention,has_negation,length_bucket,is_all_caps,has_question_mark,has_multiple_sentences,has_exclamation_mark,has_quote,conf_baseline,conf_candidate_v1,conf_candidate_v2,pred_baseline,pred_candidate_v1,pred_candidate_v2
0,0,"""Fatty Kim The Third"" 😭😭😭",neutral,3,False,False,False,0-50,False,False,False,False,True,0.486252,0.978987,0.929149,neutral,neutral,neutral
1,1,"""Focusing on [alt rightists'] respectability.....",neutral,0,False,False,False,51-100,False,False,True,False,True,0.573571,0.999728,0.832733,negative,neutral,negative
2,2,"""Kim Fatty the Third""",negative,0,False,False,False,0-50,False,False,False,False,True,0.849732,0.937709,0.910832,neutral,neutral,neutral
3,3,"""We have lost everything"": Syrians return to r...",neutral,0,True,True,False,51-100,False,False,False,False,True,0.751955,0.994244,0.815888,negative,positive,negative
4,4,"""who's the most wiped out white boy? Zac Efron...",neutral,0,False,False,False,51-100,False,True,True,True,True,0.561232,0.906538,0.476157,neutral,positive,negative


### Step 5: Compute Metrics (Accuracy + Slice Accuracy + Regression)

In [35]:
# TODO: Edit to work for your slices

#compute metrics model-wise
from sklearn.metrics import accuracy_score

def compute_accuracy(y_true, y_pred):
    return accuracy_score(list(y_true), list(y_pred))

# Overall accuracy by model (df_long: one row per (tweet, model))
overall = df_long.groupby("model").apply(
    lambda g: compute_accuracy(g["label"], g["pred"]),
    include_groups=False
)

# Slice accuracy table (uses df_long masks)
slice_table = wandb.Table(columns=["slice", "model", "accuracy"])
slice_metrics = {}

for slice_name, mask in get_slices(df_long).items():
    slice_metrics[slice_name] = {}
    for model_name, g in df_long[mask].groupby("model"):
        acc = float(compute_accuracy(g["label"], g["pred"]))
        slice_table.add_data(slice_name, model_name, acc)
        slice_metrics[slice_name][model_name] = acc

In [36]:
# TODO: Edit to work for your slices


# Regression-aware evaluation (df_eval: one row per tweet, all model outputs)
# A regression is when a candidate gets something wrong that the baseline got right.
# BASELINE is defined with MODELS in Step 1.

# Ensure ex_id exists (safe even if it already exists)
df_long = df_long.copy()
if "ex_id" not in df_long.columns:
    df_long["ex_id"] = df_long.groupby(["text", "label"]).ngroup()

assert len(META_COLS) >= 5
df_eval = (
    df_long.pivot_table(
        index=["ex_id", "text", "label"] + META_COLS,
        columns="model",
        values=["pred", "conf"],
        aggfunc="first"
    )
    .reset_index()
)

# Flatten column names (pred_baseline, conf_candidate_v1, etc.)
df_eval.columns = ["_".join([c for c in col if c]).strip("_") for col in df_eval.columns]

if f"pred_{BASELINE}" not in df_eval.columns:
    raise KeyError(f"baseline predictions missing from df_eval (need pred_{BASELINE})")

CANDIDATES = [
    m for m in MODELS
    if m != BASELINE and f"pred_{m}" in df_eval.columns
]

df_eval["baseline_correct"] = df_eval[f"pred_{BASELINE}"] == df_eval["label"]

regression_rate = {}
improvement_rate = {}
conf_reg_rate = {}

for cand in CANDIDATES:
    df_eval[f"{cand}_correct"] = df_eval[f"pred_{cand}"] == df_eval["label"]
    df_eval[f"{cand}_regressed"] = df_eval["baseline_correct"] & ~df_eval[f"{cand}_correct"]
    df_eval[f"{cand}_improved"] = ~df_eval["baseline_correct"] & df_eval[f"{cand}_correct"]
    df_eval[f"{cand}_both_wrong"] = ~df_eval["baseline_correct"] & ~df_eval[f"{cand}_correct"]
    df_eval[f"{cand}_both_correct"] = df_eval["baseline_correct"] & df_eval[f"{cand}_correct"]
    df_eval[f"{cand}_confident_regression"] = (
        df_eval[f"{cand}_regressed"] & (df_eval[f"conf_{cand}"] >= 0.8)
    )
    regression_rate[cand] = float(df_eval[f"{cand}_regressed"].mean())
    improvement_rate[cand] = float(df_eval[f"{cand}_improved"].mean())
    conf_reg_rate[cand] = float(df_eval[f"{cand}_confident_regression"].mean())
    print(cand, "regression rate:", regression_rate[cand])
    print(cand, "improvement rate:", improvement_rate[cand])
    print(cand, "confident regression rate:", conf_reg_rate[cand])

candidate_v1 regression rate: 0.408
candidate_v1 improvement rate: 0.108
candidate_v1 confident regression rate: 0.382
candidate_v2 regression rate: 0.08
candidate_v2 improvement rate: 0.098
candidate_v2 confident regression rate: 0.004


In [37]:
# TODO: Edit to work with your slices

# Define slices on df_eval (must use columns that exist in df_eval)
def get_slices_eval(df_any):
    slices = dict(get_slices(df_any))
    slices["long_tweets"] = df_any["length_bucket"].astype(str).isin(["201-1000", "1001+"])
    return slices

# Slice-level regression metrics table (one row per slice × candidate × metric)
reg_table = wandb.Table(columns=["slice", "candidate", "metric", "value"])
reg_metrics = {}

for slice_name, mask in get_slices_eval(df_eval).items():
    g = df_eval[mask]
    if len(g) == 0:
        continue

    reg_metrics[slice_name] = {}
    for cand in CANDIDATES:
        reg = float(g[f"{cand}_regressed"].mean())
        imp = float(g[f"{cand}_improved"].mean())
        conf_reg = float(g[f"{cand}_confident_regression"].mean())

        reg_table.add_data(slice_name, cand, "regression_rate", reg)
        reg_table.add_data(slice_name, cand, "improvement_rate", imp)
        reg_table.add_data(slice_name, cand, "confident_regression_rate", conf_reg)

        reg_metrics[slice_name][cand] = {
            "regression_rate": reg,
            "improvement_rate": imp,
            "conf_reg_rate": conf_reg
        }

# Step 6 — #TODO: Log to W&B & Analyse Slices
# (Make sure PROJECT/ENTITY/RUN_NAME exist from Step 1)

In [38]:
# Step 6: Log to W&B (PROJECT / ENTITY / RUN_NAME come from Step 1)

run = wandb.init(project=PROJECT, entity=ENTITY, name=RUN_NAME)
wandb.log({"predictions_table": wandb.Table(dataframe=df_long)})
wandb.log({"slice_metrics": slice_table})
wandb.log({"regression_metrics": reg_table})
wandb.log({
    "df_eval": wandb.Table(dataframe=df_eval)
})
for model_name, acc in overall.items():
    wandb.summary[f"{model_name}_accuracy"] = float(acc)
for cand, rate in regression_rate.items():
    wandb.summary[f"{cand}_regression_rate"] = rate
    wandb.summary[f"{cand}_improvement_rate"] = improvement_rate[cand]
    wandb.summary[f"{cand}_confident_regression_rate"] = conf_reg_rate[cand]

print("W&B run URL:", getattr(run, "url", None) or getattr(run, "get_url", lambda: None)())

run.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


W&B run URL: https://wandb.ai/autumnq-carnegie-mellon-university/mlip-lab4-slices-2026/runs/lpclbcfa


baseline_accuracy,0.698
candidate_v1_accuracy,0.398
candidate_v1_confident_regression_rate,0.382
candidate_v1_improvement_rate,0.108
candidate_v1_regression_rate,0.408
candidate_v2_accuracy,0.716
candidate_v2_confident_regression_rate,0.004
candidate_v2_improvement_rate,0.098
candidate_v2_regression_rate,0.08


### Instructions: Exploring Slice-Based Evaluation in W&B

# Purpose
In this lab, you are evaluating a candidate sentiment model to decide whether it should replace an existing baseline (production) model.
You have already:
  - scored every model in MODELS on the same 500 tweets
  - logged predictions, confidence scores, and metadata to W&B
  - created metadata that allows you to slice the data
The most important goal is to understand when and why models behave differently.
Overall accuracy alone is often misleading.

# What to do in W&B
1. Open your W&B run
  - Click the project link and open the latest run.
2. Explore the predictions table
  - Go to the Tables tab and open predictions_table.
  - Each row is one tweet × one model.
3. Create and analyze slices (most important)
  - Use filters to create meaningful slices 
    (e.g., negation, emojis, hashtags, long tweets).
  - For each slice:
    - Compare baseline vs candidate performance.
    - Compare slice accuracy to overall accuracy.
    - Inspect a few misclassified examples to identify patterns.
4. Visualize slice performance
  - Open slice_metrics.
  - Create bar charts comparing baseline vs candidate accuracy for at least two slices.
5. Discuss your findings with the TA
  - Explain why slicing reveals issues that overall accuracy hides.
  - Say whether the candidate model should be deployed and why.


#### Step 6b — Slice accuracy: baseline vs candidate

The next three cells do the three things the instructions above ask for, in code rather than
by clicking in the W&B UI:

1. build the baseline-vs-candidate accuracy comparison across every slice,
2. read the tweets the candidate got wrong on its **worst** slice, looking for a pattern,
3. log the comparison table and the bar charts to W&B.


In [40]:
# Step 6b.1 — Baseline vs candidate accuracy on every slice.
# df_eval is one row per tweet with pred_<model> columns, so n is unambiguous per slice.

SERIES = [BASELINE] + CANDIDATES  # column order for the chart

rows = []
for slice_name, mask in get_slices_eval(df_eval).items():
    g = df_eval[mask]
    if len(g) == 0:
        continue
    row = {"slice": slice_name, "n": int(len(g))}
    for m in SERIES:
        row[m] = float((g[f"pred_{m}"] == g["label"]).mean())
    rows.append(row)

# The "all tweets" row is the overall accuracy the slices are supposed to be compared against.
overall_row = {"slice": "ALL TWEETS", "n": int(len(df_eval))}
for m in SERIES:
    overall_row[m] = float((df_eval[f"pred_{m}"] == df_eval["label"]).mean())

slice_acc = pd.DataFrame(rows)
PRIMARY_CANDIDATE = CANDIDATES[0]
slice_acc["delta"] = slice_acc[PRIMARY_CANDIDATE] - slice_acc[BASELINE]
# Worst candidate slice first — that is the slice we read tweets from in the next cell.
slice_acc = slice_acc.sort_values(PRIMARY_CANDIDATE).reset_index(drop=True)
# Pick the worst slice from those big enough to trust: a rate on <30 tweets is noise.

overall_row["delta"] = overall_row[PRIMARY_CANDIDATE] - overall_row[BASELINE]
slice_acc_display = pd.concat([slice_acc, pd.DataFrame([overall_row])], ignore_index=True)

WORST_SLICE = slice_acc.loc[slice_acc["n"] >= 30, "slice"].iloc[0]
print(f"worst slice for {PRIMARY_CANDIDATE}: {WORST_SLICE}")
print(f"slices with n < 30 are noise and are skipped by the Step 8 gate: "
      f"{sorted(slice_acc.loc[slice_acc['n'] < 30, 'slice'])}")

slice_acc_display.style.format({**{m: "{:.1%}" for m in SERIES}, "delta": "{:+.1%}"})


worst slice for candidate_v1: has_mention
slices with n < 30 are noise and are skipped by the Step 8 gate: ['emoji_gt0']


,slice,n,baseline,candidate_v1,candidate_v2,delta
0,has_mention,208,69.7%,27.4%,68.8%,-42.3%
1,has_negation,84,66.7%,28.6%,70.2%,-38.1%
2,has_question_mark,58,65.5%,29.3%,62.1%,-36.2%
3,has_quote,156,73.7%,32.7%,69.9%,-41.0%
4,is_all_caps,89,68.5%,36.0%,76.4%,-32.6%
5,has_multiple_sentences,117,66.7%,40.2%,64.1%,-26.5%
6,has_hashtag,193,66.8%,46.1%,71.5%,-20.7%
7,emoji_gt0,29,75.9%,51.7%,79.3%,-24.1%
8,has_exclamation_mark,62,67.7%,53.2%,69.4%,-14.5%
9,ALL TWEETS,500,69.8%,39.8%,71.6%,-30.0%


In [41]:
# Step 6b.2 — Read the tweets the candidate got wrong on its worst slice.
# The question is not "is the score bad" but "do these failures share a shape".

worst_mask = get_slices_eval(df_eval)[WORST_SLICE]
worst = df_eval[worst_mask]
wrong = worst[worst[f"pred_{PRIMARY_CANDIDATE}"] != worst["label"]]

# What is the candidate saying instead of the right answer?
print(f"{WORST_SLICE}: n={len(worst)}, {PRIMARY_CANDIDATE} wrong on {len(wrong)}")
print("\ntrue label (rows) vs candidate prediction (cols):")
print(pd.crosstab(worst["label"], worst[f"pred_{PRIMARY_CANDIDATE}"]))

print(f"\nshare predicted 'positive': "
      f"{(worst[f'pred_{PRIMARY_CANDIDATE}'] == 'positive').mean():.1%} on this slice "
      f"vs {(df_eval[f'pred_{PRIMARY_CANDIDATE}'] == 'positive').mean():.1%} overall "
      f"vs {(df_eval['label'] == 'positive').mean():.1%} true positives overall")
print(f"mean confidence when wrong on this slice: {wrong[f'conf_{PRIMARY_CANDIDATE}'].mean():.3f}")

worst_errors = (
    wrong.loc[:, ["text", "label", f"pred_{BASELINE}", f"pred_{PRIMARY_CANDIDATE}",
                  f"conf_{PRIMARY_CANDIDATE}"]]
    .rename(columns={f"pred_{BASELINE}": "baseline_pred",
                     f"pred_{PRIMARY_CANDIDATE}": "candidate_pred",
                     f"conf_{PRIMARY_CANDIDATE}": "candidate_conf"})
    .sort_values("candidate_conf", ascending=False)
)

with pd.option_context("display.max_colwidth", 120):
    display(worst_errors.head(12))


has_mention: n=208, candidate_v1 wrong on 151

true label (rows) vs candidate prediction (cols):
pred_candidate_v1  negative  neutral  positive
label                                         
negative                  9        0        65
neutral                   5        5        79
positive                  0        2        43

share predicted 'positive': 89.9% on this slice vs 56.0% overall vs 22.0% true positives overall
mean confidence when wrong on this slice: 0.983


,text,label,baseline_pred,candidate_pred,candidate_conf
119,@user D.Rose taking a page out of Tim Duncan's book off the glass in the clutch #KNICKSonMSG,neutral,neutral,positive,1.000000
89,"@user @user @user UK great Antoine Walker on if his 96' team could stop Bam: ""Ask Tim Duncan about us.""#BBN",neutral,neutral,positive,1.000000
81,@user @user @user @user Poorer states than Alabama have it because they accepted Medicaid expansion,neutral,neutral,positive,1.000000
185,@user only Marine le Pen seems pleased! Very concerning for Europe. Hope some heads start to really work in Brussels!,neutral,neutral,positive,1.000000
138,@user In august2008.puppet Saakashvili attacked South Ossetia and Russian peacekeepers.South Ossetia will be i…,negative,negative,positive,1.000000
85,@user @user @user America was saved. Wake the hell up. You want 2M refugees? Global gov?TPP?Learn the facts goto col...,negative,negative,positive,1.000000
148,@user Stop Time Tonight,neutral,neutral,positive,1.000000
98,@user @user I hope this isn't another attempt to get a 300 hitter from the National League who now hits 220 Tai wins...,negative,negative,positive,1.000000
195,"@user tried to drink, but just allergies so I sat there congested and bored watching David Blaine",negative,negative,positive,1.000000
109,"@user @user oh, and Hillary most likely didn't win the popular vote, considering how many illegal votes were casted",negative,negative,positive,1.000000


In [42]:
# Step 6b.3 — Log the comparison table and the bar charts to W&B (code, not UI clicks).
# wandb/bar/v0 takes one label column and one value column, so baseline-vs-candidate is drawn
# as paired bars: the label carries the model, and sorting keeps each slice's pair adjacent.

chart_run = wandb.init(project=PROJECT, entity=ENTITY,
                       name=f"{RUN_NAME}-slice-charts", job_type="analysis")

# Wide table: one row per slice, one column per model. Lets the W&B UI regroup it too.
wandb.log({"slice_accuracy_comparison": wandb.Table(dataframe=slice_acc_display)})

# 1. The comparison chart: one bar per (slice, model).
paired = wandb.Table(columns=["slice_model", "accuracy"])
for _, r in slice_acc.iterrows():
    for m in SERIES:
        paired.add_data(f"{r['slice']} · {m}", float(r[m]))

# 2. The same story as one bar per slice: how far the candidate moved the slice.
delta = wandb.Table(columns=["slice", "accuracy_delta"])
for _, r in slice_acc.iterrows():
    delta.add_data(f"{r['slice']} (n={int(r['n'])})", float(r["delta"]))

wandb.log({
    "slice_accuracy_baseline_vs_candidate": wandb.plot.bar(
        paired, "slice_model", "accuracy",
        title=f"Accuracy by slice — {BASELINE} vs {PRIMARY_CANDIDATE}"),
    "slice_accuracy_delta": wandb.plot.bar(
        delta, "slice", "accuracy_delta",
        title=f"{PRIMARY_CANDIDATE} minus {BASELINE} accuracy (negative = regression)"),
    "worst_slice_errors": wandb.Table(dataframe=worst_errors.reset_index(drop=True)),
})

print("W&B run URL:", getattr(chart_run, "url", None))
chart_run.finish()


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


W&B run URL: https://wandb.ai/autumnq-carnegie-mellon-university/mlip-lab4-slices-2026/runs/86r07sj4


In [ ]:
# Step 6b.4 — 1–2 sentence takeaway per slice, written after reading the tweets above.

saved_slice_notes = {
    "has_mention": (
        "Worst slice by far: baseline 69.7% -> candidate 27.4% (n=208), and the candidate calls "
        "90% of these tweets positive while only 22% of them are. The @user token itself is the "
        "trigger, not the sentiment — tweets that *start* with @user are labelled positive 94.9% "
        "of the time at a mean confidence of 0.99, so angry political replies come back "
        "'positive' with p=0.9999."
    ),
    "has_quote": (
        "Second-biggest absolute drop: 73.7% -> 32.7% (n=156). Quoted text is mostly neutral "
        "reporting (14% is truly positive) and the candidate's positive default overwrites it, "
        "taking neutral recall from 62% down to 27%."
    ),
    "has_negation": (
        "66.7% -> 28.6% (n=84). Only 6% of negated tweets are positive, yet the candidate calls "
        "52% of them positive — it is not misreading the negation so much as ignoring the text, "
        "and 65% of the slice is *confidently* wrong (conf >= 0.9)."
    ),
    "has_question_mark": (
        "65.5% -> 29.3% (n=58). Same shape as negation: 5% of questions are truly positive, the "
        "candidate says positive for 52% of them, and neutral recall collapses from 64% to 18%."
    ),
    "is_all_caps": (
        "68.5% -> 36.0% (n=89). Shouting is the one cue that does *not* flip the candidate to "
        "positive as hard as the mention cue does (51% positive predictions), so this slice sits "
        "mid-table — but 38 of the 61 tweets the baseline got right here still come back wrong."
    ),
    "has_multiple_sentences": (
        "66.7% -> 40.2% (n=117). Smaller drop than the single-cue slices, which argues the "
        "problem is not 'the model reads only the first clause' — longer tweets just give the "
        "positive prior more real text to compete with."
    ),
    "has_hashtag": (
        "66.8% -> 46.1% (n=193). Milder than mentions even though both are Twitter markup, so "
        "the candidate's failure is specific to the mention token rather than to markup in "
        "general — this is the comparison that isolates the cause."
    ),
    "has_exclamation_mark": (
        "Smallest drop: 67.7% -> 53.2% (n=62). Not because the candidate handles exclamations "
        "well, but because 42% of this slice really is positive, so a model that guesses positive "
        "gets graded generously — a reminder that a healthy-looking slice score can be luck."
    ),
    "emoji_gt0": (
        "75.9% -> 51.7% but n=29, under the 30-example floor, so the Step 8 gate skips it and so "
        "should we — the same 42% positive base rate as the exclamation slice explains the score."
    ),
    "long_tweets": (
        "Empty slice: n=0, because tweets top out at 200 characters and the bucket starts at 201, "
        "so it tests nothing. Either drop it or re-cut the buckets around the real range (the "
        "longest band that actually has tweets is 101-200, with 218 of the 500)."
    ),
}

# Across-slice read: the candidate predicts 'positive' for 56% of all 500 tweets when the true
# positive rate is 22%, and 53% of every tweet is confidently wrong (conf >= 0.9). The slices do
# not each expose a different bug — they rank almost perfectly by how closely the slice's true
# positive rate matches that stuck prior, with has_mention the one genuine outlier where the
# token itself drives the collapse. Overall accuracy 69.8% -> 39.8%: do not ship candidate_v1.

notes_df = (
    pd.DataFrame({"slice": list(saved_slice_notes), "note": list(saved_slice_notes.values())})
    .merge(slice_acc_display[["slice", "n", BASELINE, PRIMARY_CANDIDATE, "delta"]],
           on="slice", how="left")
    .loc[:, ["slice", "n", BASELINE, PRIMARY_CANDIDATE, "delta", "note"]]
)
with pd.option_context("display.max_colwidth", 200):
    display(notes_df)


### Step 7 - Targeted stress testing with LLMs

TODO: 
In this step, you will use a Large Language Model (LLM) to generate test cases that specifically target a weakness you observed during slicing.

What to do:
1. Choose one slice where you noticed poor performance, regressions, or surprising behavior.
2. Write a short hypothesis (1–2 sentences) explaining why the model might struggle on this slice. Example:
“The model struggles with tweets that use slang and sarcasm.”
3. Use an LLM to generate 10 test cases designed to test this hypothesis.
These can include:
    - subtle or ambiguous cases
    - difficult or adversarial cases
    - small wording changes that affect sentiment
4. Re-run every model in `MODELS` on the generated test cases (helper code given below).
5. Briefly describe what you observed to the TA:
    - Did the same failures appear again?
    - notice any new failure patterns?
    - would this affect your confidence in deploying the model?

Your input can be in the following format:

> Examples:
> - @user @user That’s coming, but I think the victims are going to be Medicaid recipients.
> - I think I may be finally in with the in crowd #mannequinchallenge  #grads2014 @user
> 
> Generate more tweets using slangs.

Use our provided GPTs to start the task: [llm-based-test-case-generator](https://chatgpt.com/g/g-982cylVn2-llm-based-test-case-generator). If you do not have access to GPTs, use the plain ChatGPT or other LLM providers you have access to instead.

**This is the one step that needs internet.** Steps 1–6 read saved predictions, but your tweets are new, so the cell below downloads each model in `MODELS` and runs it — about 1 GB for `baseline` + `candidate_v1`, on the first run only. Do this step while `INCLUDE_CANDIDATE_V2 = False`. If you come back and re-run it after Task G, it will also pull `candidate_v2`, which is a **1.7 GB** download and roughly 6× slower per tweet.

In [27]:
# TODO: Paste your 10 generated tweets here:
generated_slice_description = "The model seems to be biased towards predicting positive sentiment when it sees the '@' symbol, which is often used in replies or mentions on Twitter."

generated_cases = [
    "@user This is exactly what I needed today. Thank you!",
    "@user Your customer service is absolutely terrible. Never again.",
    "@user @team The meeting starts at 3 PM, right?",
    "@user I expected the update to break everything, but this is actually pretty nice.",
    "@user Sure, the new design is *interesting*. Not sure that's a compliment.",
    "@user @user Just passing along the information from today's announcement.",
    "@user I don't mind waiting a little longer for this.",
    "@user I don't like waiting a little longer for this.",
    "@user @user The presentation was surprisingly helpful.",
    "@user @user The presentation was surprisingly unhelpful."
]

intended_labels = [
    "positive",
    "negative",
    "neutral",
    "positive",
    "negative",
    "neutral",
    "positive",
    "negative",
    "positive",
    "negative"
]
assert len(generated_cases) == 10 and len(intended_labels) == 10

In [28]:
#Helper code to run models on synthetic test cases:

def run_on_generated_tests(texts, models=MODELS):
    rows = []
    for model_name, spec in models.items():
        hf_id = spec["hf_id"] if isinstance(spec, dict) else spec
        clf = pipeline(
            "text-classification",
            model=hf_id,
            truncation=True,
            framework="pt",
            device=-1
        )
        for t in texts:
            out = clf(t)[0]
            rows.append({
                "text": t,
                "model": model_name,
                "pred": HF_LABEL_MAP.get(out["label"], out["label"]),
                "conf": float(out["score"])
            })
    return pd.DataFrame(rows)


In [29]:
# Skip live HF inference until 10 real tweets are pasted (keeps the notebook offline-runnable).
if not any(str(t).strip() for t in generated_cases):
    generated_df = pd.DataFrame(columns=["text", "model", "pred", "conf", "intended", "correct"])
    print("TODO: paste 10 generated tweets and intended labels, then re-run this cell.")
else:
    generated_df = run_on_generated_tests(generated_cases)
    intended = pd.DataFrame({"text": generated_cases, "intended": intended_labels})
    generated_df = generated_df.merge(intended, on="text", how="left")
    generated_df["correct"] = generated_df["pred"] == generated_df["intended"]
    print(generated_slice_description)
    print(generated_df.groupby("model")["correct"].mean())
generated_df

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1637.15it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 1670.35it/s]
[transformers] GPT2ForSequenceClassification LOAD REPORT from: LYTinn/finetuning-sentiment-model-tweet-gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The model seems to be biased towards predicting positive sentiment when it sees the '@' symbol, which is often used in replies or mentions on Twitter.
model
baseline        0.9
candidate_v1    0.5
Name: correct, dtype: float64


,text,model,pred,conf,intended,correct
0,@user This is exactly what I needed today. Tha...,baseline,positive,0.982298,positive,True
1,@user Your customer service is absolutely terr...,baseline,negative,0.952503,negative,True
2,"@user @team The meeting starts at 3 PM, right?",baseline,neutral,0.953853,neutral,True
3,@user I expected the update to break everythin...,baseline,positive,0.750018,positive,True
4,"@user Sure, the new design is *interesting*. N...",baseline,positive,0.692984,negative,False
5,@user @user Just passing along the information...,baseline,neutral,0.942388,neutral,True
6,@user I don't mind waiting a little longer for...,baseline,positive,0.797433,positive,True
7,@user I don't like waiting a little longer for...,baseline,negative,0.918352,negative,True
8,@user @user The presentation was surprisingly ...,baseline,positive,0.984027,positive,True
9,@user @user The presentation was surprisingly ...,baseline,negative,0.929069,negative,True


In [21]:
# OPTIONAL: Log synthetic test cases to W&B
if len(generated_df):
    synth_run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name="synthetic-tests",
        job_type="stress-test",
    )
    wandb.log({"synthetic_tests": wandb.Table(dataframe=generated_df)})
    print("W&B run URL:", getattr(synth_run, "url", None))
    synth_run.finish()
else:
    print("Skipping W&B log: no synthetic rows yet.")

Skipping W&B log: no synthetic rows yet.


### Step 8 — Exploration vs enforcement (regression gate)

Steps 1–7 were **exploration**: you sliced in a notebook, stared at W&B, and generated synthetic tweets. That is how you *find* hypotheses.

This step is **enforcement**. You encode the slices you care about as tests with **thresholds chosen before you score the next candidate**. Freeze `tests/manifest.yaml` using the baseline (and what you learned from `candidate_v1`). Then run the **same** suite on `candidate_v2` without editing the manifest. A slice with fewer than 30 examples is skipped — a rate estimated on 5 tweets is noise, not a gate.

Until Task G, leave `INCLUDE_CANDIDATE_V2 = False` in Step 1 so W&B is baseline vs `candidate_v1` only. Predictions for v2 are already in `tweets.csv`; hiding them is so you do not retune the gate.

Edit `tests/manifest.yaml` first (slice name, threshold, rationale, and the p50 latency cap). Then run pytest from the repo root.

The model runs on a live moderation stream, so the product budget is **p50 under 50 ms per tweet on one CPU thread**. Timings are already measured and saved in `tweets.csv`, so pytest reads them instead of re-timing on your laptop.

Manifest format:

```yaml
slices:
  - slice: has_negation
    threshold: 0.60
    rationale: ""
latency:
  p50_ms_max: 50
  rationale: ""
```


In [30]:
# Run these from the repo root *after* you freeze thresholds in tests/manifest.yaml.
# Do not retune the manifest to make a candidate pass.

print("MODEL=baseline pytest tests/ -v")
print("MODEL=candidate_v1 pytest tests/ -v")
# Task G — same suite, do not edit the manifest:
print("MODEL=candidate_v2 pytest tests/ -v")

MODEL=baseline pytest tests/ -v
MODEL=candidate_v1 pytest tests/ -v
MODEL=candidate_v2 pytest tests/ -v


In [31]:
# Load the pytest session CSV and log it to W&B as gate_results.
from pathlib import Path

model = os.environ.get("MODEL", "candidate_v1")
gate_path = Path(f"tests/gate_results_{model}.csv")
if not gate_path.exists():
    print(f"No {gate_path} yet. Run: MODEL={model} pytest tests/ -v")
    gate_df = pd.DataFrame(columns=["test_id", "observed", "threshold", "outcome"])
else:
    gate_df = pd.read_csv(gate_path)
    gate_run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name=f"gate-results-{model}",
        job_type="gate",
    )
    wandb.log({"gate_results": wandb.Table(dataframe=gate_df)})
    print("W&B run URL:", getattr(gate_run, "url", None))
    gate_run.finish()
gate_df

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


W&B run URL: https://wandb.ai/autumnq-carnegie-mellon-university/mlip-lab4-slices-2026/runs/nkn7gpbi


,test_id,observed,threshold,outcome
0,slice::has_mention,0.274038,0.690000,failed
1,vs_baseline::has_mention,0.274038,0.677115,failed
2,latency::p50,28.829521,50.000000,passed
